# Spin the black circle II

Figures of [arXiv:2603.28982](https://arxiv.org/abs/2603.28982), numbered as in the paper.
Figure 7 (quasi-circular instantaneous fluxes) compares against frequency-domain data that is not ours to redistribute, so we do not reproduce it here.


## Imports and setup

In [ ]:
import numpy as np
from plot_settings import plt, keys_to_labels, colors_orbit_types, colors
from matplotlib.colors import ListedColormap
import json
from itertools import groupby
import matplotlib.ticker as mticker
from plot_settings import lighten_color

# local imports
import plottk
import utils as ut

In [ ]:
with open("../dat/data.txt", "r") as f:
    all_data_raw = np.genfromtxt(f, names=True)
    names = all_data_raw.dtype.names

all_data = {}
for name in names:
    all_data[name] = all_data_raw[name]

# data release folder
basedir = "../dat/data_release/"

## Figure 1

In [ ]:
all_data["E"] = 10 ** all_data["logE"]
all_data["Pphi"] = 10 ** all_data["logPphi"]

for name in ("dE", "dJ", "dEavg", "dJavg"):
    all_data["log" + name] = np.log10(abs(all_data_raw[name]))

names = [
    "a",
    "E",
    "Pphi",
    "logdE",
    "logdJ",
    "rmin",
    "e",
    "p",
    "logdEavg",
    "logdJavg",
]

# set the bins for the histograms based on max and min of each parameter
bins = {
    name: np.linspace(np.min(all_data[name]), np.max(all_data[name]), 10)
    for name in names
}

# divide the dataset into three subsets based on the value of kind
kinds = np.unique(all_data_raw["kind"])
markers = [colors_orbit_types[k][1] for k in colors_orbit_types]
ocolors = [colors_orbit_types[k][0] for k in colors_orbit_types]
data_subsets = [
    {name: all_data[name][all_data_raw["kind"] == kind] for name in names}
    for kind in kinds
]

In [ ]:
names = ["a", "E", "Pphi", "logdE", "logdJ", "rmin"]
nrows = len(names)
fig, axs = plt.subplots(nrows, nrows, figsize=(9, 9))

lower_pairs = [(i, j) for i in range(nrows) for j in range(i)]

for subset, marker, color in zip(data_subsets, markers, ocolors):
    # make histograms on the diagonal
    for i in range(nrows):
        axs[i, i].hist(
            subset[names[i]],
            bins=bins[names[i]],
            color=color,
            alpha=0.5,
            density=False,
        )

    # make scatter plots on the lower triangle
    for i, j in lower_pairs:
        axs[i, j].scatter(
            subset[names[j]],
            subset[names[i]],
            s=30,
            marker=marker,
            color=color,
            alpha=0.7,
        )

for i in range(nrows):
    # deactivate upper triangle and style lower triangle
    for j in range(i + 1, nrows):
        axs[i, j].axis("off")
    for j in range(i):
        axs[i, j].grid(color="lightgray", linestyle=":", linewidth=0.7)

# labels only on the left and bottom plots
for i, name in enumerate(names):
    if i > 0:
        axs[i, 0].set_ylabel(keys_to_labels[name], fontsize=15)
    axs[-1, i].set_xlabel(keys_to_labels[name], fontsize=15)

# remove xtick labels on lower triangle plots except for the bottom row
for i in range(nrows - 1):
    for j in range(i + 1):
        axs[i, j].set_xticklabels([])

# remove yticks on lower triangle plots except for the left column
for i in range(nrows):
    for j in range(1, i + 1):
        axs[i, j].set_yticklabels([])

# remove yticks on diagonal plots
for i in range(nrows):
    axs[i, i].set_yticks([])

# manual legend in position outside the plot
handles = [
    plt.Line2D([], [], marker=marker, color=color, linestyle="", label=kind)
    for marker, color, kind in zip(markers, ocolors, ["Circular", "Eccentric", "Hyperbolic"])
]
fig.legend(
    handles=handles,
    fontsize=15,
    fancybox=False,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.75, 0.8),
)

# reduce spacing from top/bottom and left/right
plt.subplots_adjust(
    left=0.1, right=0.95, top=0.93, bottom=0.07, hspace=0.1, wspace=0.1
)

## Figure 2

In [ ]:
# load the three simulations
configs = [
    "circular/teuk_a0.0000_r04.0000_circ",
    "eccentric/teuk_a0.0000_e0.8000_p09.0000",
    "hyperbolic/teuk_a0.8000_E1.0350_L8.0000_r0100.0000",
]
kinds = ["Circular", "Eccentric", "Hyperbolic"]

fig, ax = plt.subplots(2, 1, sharex=True)
figd, axd = plt.subplots(subplot_kw={"projection": "polar"})

for cfg, kind in zip(configs, kinds):
    sim = basedir + cfg
    sim_name = sim.split("/")[-1]
    abh = float(sim_name.split("_")[1][1:])
    ptk = plottk.PlotterTKO(a=abh, path_dyn=sim, path_tk=sim)
    res = max(ptk.sims, key=lambda k: len(ptk.sims[k]))  # resolution with the most m modes
    ptk.load_dyn()
    hflx = ptk.integrate_hflx(nx=res[0], ny=res[1], cfl=res[2])

    col = colors_orbit_types[kind.lower()][0]

    axd.plot(ptk.dyn["ph"], ptk.dyn["r"], color=col)

    t_num = ptk.hflx["t"] + ut.adv_time_in(ptk.a, ptk.dyn["r"][0])
    t_norm = t_num / np.max(t_num)

    ax[0].semilogy(t_norm, abs(hflx["sum"]["E"]["dy"]), color=col, label=kind)
    ax[1].semilogy(t_norm, abs(hflx["sum"]["J"]["dy"]), color=col)

# fluxes settings
ax[0].set_ylim(1e-19, 1e-0)
ax[1].set_ylim(1e-15, 1e-0)
ax[1].set_xlim(0.1, 1)
ax[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
    ncol=3,
    frameon=False,
    fancybox=False,
)

for tax in ax:
    tax.grid(color="lightgray", linestyle=":", linewidth=0.7)

ax[0].set_ylabel(r"$\dot{M}$")
ax[1].set_ylabel(r"$\dot{S}$")
ax[1].set_xlabel(r"$t/t_{\rm max}$")

# dynamics settings
axd.set_ylim(0, 50)
axd.set_ylabel(r"$r$")
axd.set_xlabel(r"$\varphi$")
plt.show()

## Figure 3

In [ ]:
data_subsets = [all_data_raw[all_data_raw["kind"] == orbit_type] for orbit_type in (0, 1, 2)]


def select_fields(dataset, field_names, mask):
    return {name: dataset[name][mask] for name in field_names}


def sorted_series(dataset, x_key, *y_keys):
    order = np.argsort(dataset[x_key])
    x_sorted = dataset[x_key][order]
    y_sorted = [dataset[key][order] for key in y_keys]
    return x_sorted, y_sorted

In [ ]:
# eccentric
eccentric_set = data_subsets[1]
names = ["a", "p", "e", "dEavg", "dJavg"]
line_styles = ["-", "--", ":", "-."]
point_markers = ["o", "s", "D", "^", "v"]

a0_values = np.unique(eccentric_set["a"])
p0_values = np.unique(eccentric_set["p"])

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 6))

# use only a0 values that have at least 2 different e values
a0_values = [
    a0
    for a0 in a0_values
    if len(np.unique(eccentric_set["e"][eccentric_set["a"] == a0])) >= 2 and a0 != 0.1
]
p0_values = [
    p0
    for p0 in p0_values
    if len(np.unique(eccentric_set["e"][eccentric_set["p"] == p0])) >= 2 and p0 <= 11
]

for a0, color in zip(a0_values, colors):
    subset_a0 = select_fields(eccentric_set, names, eccentric_set["a"] == a0)

    for p0, ls, mrk in zip(p0_values, line_styles, point_markers):
        subset = select_fields(subset_a0, names, subset_a0["p"] == p0)
        if subset["dEavg"].size < 2:
            continue

        e_sorted, (dE_sorted, dJ_sorted) = sorted_series(subset, "e", "dEavg", "dJavg")
        ax[0].plot(e_sorted, dE_sorted, color=color, linestyle=ls, marker=mrk)
        ax[1].plot(e_sorted, dJ_sorted, color=color, linestyle=ls, marker=mrk)

for tax in ax:
    tax.axhline(0.0, color="black", linestyle=":", linewidth=1.0)
    tax.set_yscale("asinh", linear_width=1.0e-9)
    tax.grid(color="lightgray", linestyle=":", linewidth=0.7)

ax[-1].set_xlabel(r"$e_0$", fontsize=15)
ax[0].set_ylabel(r"$\langle \dot{M}\rangle$", fontsize=15)
ax[1].set_ylabel(r"$\langle \dot{S}\rangle$", fontsize=15)

# legend outside of the plot
legend_handles = [
    plt.Line2D(
        [],
        [],
        color=color,
        linestyle="-",
        marker="o",
        label=rf"$\hat{{a}}={a0:.1f}$",
    )
    for a0, color in zip(a0_values, colors)
]
legend_handles_e = [
    plt.Line2D(
        [],
        [],
        color="k",
        linestyle=ls,
        marker=mrk,
        label=f"$p_0={p0:.0f}$",
    )
    for p0, ls, mrk in zip(p0_values, line_styles, point_markers)
]

first_legend = ax[0].legend(
    handles=legend_handles,
    fontsize=15,
    loc="upper left",
    bbox_to_anchor=(-0.05, 1.35),
    fancybox=False,
    ncol=2,
    frameon=False,
)
ax[0].add_artist(first_legend)
ax[0].legend(
    handles=legend_handles_e,
    fontsize=15,
    loc="upper right",
    bbox_to_anchor=(1.05, 1.35),
    fancybox=False,
    ncol=1,
    frameon=False,
)
ax[0].set_yticks([-1.0e-7, -1.0e-8, 0, 1.0e-8, 1.0e-7, 1.0e-6, 1.0e-5])
ax[1].set_yticks([-1.0e-5, -1.0e-6, -1.0e-7, -1.0e-8, 0, 1.0e-8, 1.0e-7, 1.0e-6, 1.0e-5, 1.0e-4])
ax[0].set_ylim([ax[0].get_ylim()[0], 3.0e-5])
ax[1].set_ylim([ax[1].get_ylim()[0], 3.0e-4])
plt.show()

In [ ]:
# hyperbolic
hyperbolic_set = data_subsets[2]
names = ["logE", "logPphi", "a", "dE", "dJ", "E", "J", "dE2", "dJ2"]
line_styles = [":", "-", "--", "-.", (0, (3, 1, 1, 1))]
point_markers = ["o", "s", "D", "^", "v"]

a0_values = np.unique(hyperbolic_set["a"])
pph_values = np.unique(hyperbolic_set["logPphi"])

# only pph <= 12 (less crowded)
pph_values = pph_values[10**pph_values <= 13.0]
a0_values = [
    a0
    for a0 in a0_values
    if len(np.unique(hyperbolic_set["logE"][hyperbolic_set["a"] == a0])) > 3 and a0 != 0.4
]
pph_values = [
    pph
    for pph in pph_values
    if len(np.unique(hyperbolic_set["logE"][hyperbolic_set["logPphi"] == pph])) > 3 and 10**pph != 6
]

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
for a0, color in zip(a0_values, colors):
    subset_a0 = select_fields(hyperbolic_set, names, hyperbolic_set["a"] == a0)

    for pph, ls, mrk in zip(pph_values, line_styles, point_markers):
        subset = select_fields(subset_a0, names, subset_a0["logPphi"] == pph)
        if subset["dE"].size < 2:
            continue

        E_sorted, (dJ_sorted, dE_sorted) = sorted_series(subset, "logE", "dJ", "dE")
        ax[0].semilogy(10**E_sorted, dE_sorted, color=color, linestyle=ls, marker=mrk)
        ax[1].semilogy(10**E_sorted, dJ_sorted, color=color, linestyle=ls, marker=mrk)

for tax in ax:
    tax.axhline(0.0, color="k", linestyle=":", linewidth=1.0)
    tax.grid(color="lightgray", linestyle=":", linewidth=0.7)

ax[0].set_yscale("asinh", linear_width=1.0e-13)
ax[1].set_yscale("asinh", linear_width=1.0e-11)

ax[1].set_xlabel(r"$E_0$", fontsize=17)
ax[0].set_ylabel(r"$\dot{M}_{\rm peak}$", fontsize=17)
ax[1].set_ylabel(r"$\dot{S}_{\rm peak}$", fontsize=17)

# legends outside of the plot, two separate legends for a0 and pph
legend_handles = [
    plt.Line2D(
        [],
        [],
        color=color,
        linestyle="-",
        marker="o",
        label=rf"$\hat{{a}}={a0:.1f}$",
    )
    for a0, color in zip(a0_values, colors)
]
legend_handles_pph = [
    plt.Line2D(
        [],
        [],
        color="k",
        linestyle=ls,
        marker=mrk,
        label=f"$p_{{\phi 0}}={10**pph:.0f}$",
    )
    for pph, ls, mrk in zip(pph_values, line_styles, point_markers)
]

first_legend = ax[0].legend(
    handles=legend_handles,
    fontsize=15,
    fancybox=False,
    frameon=False,
    loc="upper left",
    ncol=2,
    bbox_to_anchor=(-0.05, 1.35),
)

ax[0].add_artist(first_legend)
ax[0].legend(
    handles=legend_handles_pph,
    fontsize=15,
    fancybox=False,
    frameon=False,
    loc="upper right",
    ncol=2,
    bbox_to_anchor=(1.05, 1.35),
)
ax[0].set_yticks([-1.0e-3, -1.0e-5, -1.0e-7, -1.0e-9, -1.0e-11, 0, 1.0e-11, 1.0e-9, 1.0e-7, 1.0e-5, 1.0e-3])
ax[1].set_yticks([-1.0e-3, -1.0e-5, -1.0e-7, -1.0e-9, 0, 1.0e-9, 1.0e-7, 1.0e-5, 1.0e-3])
plt.show()

## Figure 4

In [ ]:
from matplotlib.lines import Line2D

configs = [
    "../dat/data_release/eccentric/teuk_a0.3000_e0.6500_p09.0000/",
    "../dat/data_release/hyperbolic/teuk_a0.3000_E1.0050_L4.0000_r0100.0000/",
]
kinds = ["Eccentric", "Hyperbolic"]
mode_styles = [(0, ":"), (1, "-."), (2, "--")]
zoom_min, zoom_max = 0.27, 0.43

lg_color = "black"
grid_color = "lightgray"

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(6.4, 4.8))

for sim, kind in zip(configs, kinds):
    sim_name = sim.split("/")[-2]
    abh = float(sim_name.split("teuk_a")[1].split("_")[0])
    ptk = plottk.PlotterTKO(a=abh, path_dyn=sim, path_tk=sim)
    res = max(ptk.sims, key=lambda k: len(ptk.sims[k]))  # resolution with the most m modes
    ptk.load_dyn()
    hflx = ptk.integrate_hflx(nx=res[0], ny=res[1], cfl=res[2])

    col = colors_orbit_types[kind.lower()][0]

    t_num = ptk.hflx["t"] + ut.adv_time_in(ptk.a, ptk.dyn["r"][0])
    xmax = np.max(t_num)
    t_norm = t_num / xmax
    zoom_mask = (t_norm >= zoom_min) & (t_norm <= zoom_max)

    for axis, key, label in ((ax[0], "E", kind), (ax[1], "J", None)):
        axis.plot(t_norm[zoom_mask], hflx["sum"][key]["dy"][zoom_mask], color=col, lw=2, label=label)
        for mode, ls in mode_styles:
            axis.plot(
                t_norm[zoom_mask],
                hflx[mode][key]["dy"][zoom_mask],
                color=col,
                alpha=0.8,
                linestyle=ls,
                lw=1,
            )

    omgH = ut.get_omgH(abh)
    dyn_mask = (ptk.dyn["t"] / xmax >= zoom_min) & (ptk.dyn["t"] / xmax <= zoom_max)
    j_sr = np.where(ptk.dyn["Omg"][dyn_mask] > omgH)[0]
    t_sr = [ptk.dyn["t"][dyn_mask][j_sr[0]], ptk.dyn["t"][dyn_mask][j_sr[-1]]]
    for tax in ax:
        tax.axvline(t_sr[0] / xmax, color=col, linestyle="-", linewidth=1.0)
        tax.axvline(t_sr[1] / xmax, color=col, linestyle="-", linewidth=1.0)

ax[1].set_xlim(zoom_min, zoom_max)

custom_lines = [
    Line2D([0], [0], color=lg_color, linestyle="-", lw=1, label=r"Total"),
    Line2D([0], [0], color=lg_color, linestyle=":", lw=1, label=r"$m=0$"),
    Line2D([0], [0], color=lg_color, linestyle="-.", lw=1, label=r"$m=1$"),
    Line2D([0], [0], color=lg_color, linestyle="--", lw=1, label=r"$m=2$"),
]

l1 = ax[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.3),
    ncol=2,
    frameon=False,
    fancybox=False,
)
ax[0].add_artist(l1)
ax[0].legend(
    handles=custom_lines,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.2),
    ncol=4,
    frameon=False,
    fancybox=False,
)

for tax in ax:
    tax.grid(color=grid_color, linestyle=":", linewidth=0.7)
    tax.set_yscale("asinh", linear_width=1.0e-10)
    tax.axhline(0.0, color=lg_color, linestyle="--", linewidth=1.0)

ax[0].set_ylabel(r"$\dot{M}$")
ax[1].set_ylabel(r"$\dot{S}$")
ax[1].set_xlabel(r"$t/t_{\rm max}$")

ax[0].set_yticks([-1.0e-7, -1.0e-8, -1.0e-9, 0, 1.0e-9, 1.0e-8, 1.0e-7, 1.0e-6])
ax[1].set_yticks([-1.0e-6, -1.0e-7, -1.0e-8, -1.0e-9, 0, 1.0e-9, 1.0e-8, 1.0e-7, 1.0e-6, 1.0e-5])
ax[0].set_ylim([ax[0].get_ylim()[0], 1.0e-5])
ax[1].set_ylim([ax[1].get_ylim()[0], 1.0e-4])

# adjust top and left/right margins
plt.subplots_adjust(top=0.9, left=0.15, right=0.95)
plt.show()

## Figure 5

In [ ]:
def _to_numpy_arrays(runs):
    for run in runs:
        for key, value in run.items():
            if isinstance(value, list):
                run[key] = np.array(value)


def _add_derived_quantities(runs):
    zero_specs = (
        ("t_omgH", "r_omgH", "prstar_omgH", "vr_omgH"),
        ("t_J_zero", "r_J_zero", "prstar_J_zero", "vr_J_zero"),
        ("t_E_zero", "r_E_zero", "prstar_E_zero", "vr_E_zero"),
        ("t_Er_zero", "r_Er_zero", "prstar_Er_zero", "vr_Er_zero"),
    )

    for run in runs:
        run["omgH"] = ut.get_omgH(run["a"])

        for t_key, r_key, pr_key, out_key in zero_specs:
            if t_key in run and run[t_key] is not None:
                run[out_key] = ut.get_vr(run[r_key], run[pr_key], run["L0"], run["a"])


def _sort_key(run):
    if "vr_E_zero" in run and len(run["vr_E_zero"]) > 0:
        return abs(run["vr_E_zero"][0] / run["r_E_zero"][0] / run["omg_E_zero"][0])
    return 1e6


def _compute_omega_t_over_omega_h(run, channel, model, nu, emm):
    use_p_model = model.endswith("_p")

    if channel == "S":
        if use_p_model:
            omega_t = ut.OmegaTS_p(
                run["r_J_zero"],
                run["L0"],
                run["prstar_J_zero"],
                nu,
                run["a"],
                model=model,
                emm=emm,
            )
        else:
            omega_t = ut.OmegaTS(
                run["r_J_zero"],
                run["omg_J_zero"],
                run["vr_J_zero"],
                nu,
                run["a"],
                model=model,
                emm=emm,
            )
    else:
        if use_p_model:
            omega_t = ut.OmegaTE_p(
                run["r_E_zero"],
                run["L0"],
                run["prstar_E_zero"],
                nu,
                run["a"],
                model=model,
                emm=emm,
            )
        else:
            omega_t = ut.OmegaTE(
                run["r_E_zero"],
                run["omg_E_zero"],
                run["vr_E_zero"],
                nu,
                run["a"],
                model=model,
                emm=emm,
            )
    return abs(omega_t / abs(run["omgH"]))


def _plot_panel(
    ax_panel,
    runs,
    filename,
    channel,
    nu,
    emm,
    model_specs,
    start_idx,
    xvar="vr"
):
    marker = "<" if "hyp" in filename else "X"
    omega_key = "omg_J_zero" if channel == "S" else "omg_E_zero"
    vr_key = "vr_J_zero" if channel == "S" else "vr_E_zero"
    t_zero_key = "t_J_zero" if channel == "S" else "t_E_zero"
    channel_key = "J" if channel == "S" else "E"

    valid_runs = [
        run
        for run in runs
        if run["t_omgH"] is not None
        or (run[t_zero_key] is not None and len(run[t_zero_key]) > 0)
    ]

    counts = {
        model: {"rdot_pos": [0, 0], "rdot_neg": [0, 0]} for model, _, _ in model_specs
    }

    for run_idx, run in enumerate(valid_runs, start=start_idx):

        for model, color, _ in model_specs:
            omega_t = _compute_omega_t_over_omega_h(run, channel, model, nu, emm)
            if len(omega_t) == 0:
                continue
            if xvar == "vr":
                # radial-to-angular velocity ratio at each flux zero
                xvals = np.array(
                    [
                        abs(
                            run[f"vr_{channel_key}_zero"][jz]
                            / run[f"r_{channel_key}_zero"][jz]
                            / run[f"omg_{channel_key}_zero"][jz]
                        )
                        ** 2
                        for jz in range(len(run[omega_key]))
                    ]
                )
            elif xvar == "run":
                xvals = [run_idx] * len(run[omega_key])

            sign_cmap = ListedColormap(["w", color], "sign", 2)
            ax_panel.scatter(
                xvals,
                omega_t,
                c=-np.sign(run[vr_key]),
                cmap=sign_cmap,
                marker=marker,
                edgecolors=color,
            )

            in_band = (omega_t >= 0.9) & (omega_t <= 1.1)
            rdot_pos = run[vr_key] > 0
            rdot_neg = run[vr_key] < 0
            counts[model]["rdot_pos"][0] += int(np.sum(in_band & rdot_pos))
            counts[model]["rdot_pos"][1] += int(np.sum(rdot_pos))
            counts[model]["rdot_neg"][0] += int(np.sum(in_band & rdot_neg))
            counts[model]["rdot_neg"][1] += int(np.sum(rdot_neg))

    return start_idx + len(valid_runs), counts

In [ ]:
def _ax_lims(orbit, emm):
    if orbit == "eccentric":
        return {
            (2,): (0.0, 2.0),
            (1,): (-1.0e-4, 4.0),
            (0, 1, 2): (-1.0e-4, 2.5),
        }[tuple(emm)]
    return {
        (2,): (0.0, 500.0),
        (1,): (-1.0e-4, 25),
        (0, 1, 2): (-2.0e-4, 1.3e4),
    }[tuple(emm)]


nu = 1.0e-3
model_specs = [
    ("base_p", colors[0], r"$\Omega_T$"),
    ("base", colors[1], r"$\Omega_T^{\rm dt}$"),
    ("ns_p", colors[2], r"$\Omega_T^{\rm NS}$"),
    ("ns", colors[3], r"$\Omega_T^{\rm NS, dt}$"),
]

for orbit in ["eccentric", "hyperbolic"]:
    for m_str in ["012", "1", "2"]:
        emm = [int(m) for m in m_str]
        omega_symbol = r"\Omega_T" if len(emm) > 1 else rf"\Omega_{{2 {emm[0]} T}}"
        xlim = _ax_lims(orbit, emm)
        filename = f"../dat/superradiance_{orbit}_m{m_str}_fixed_interactive_final.json"

        fig, ax = plt.subplots(2, 1, sharex=True, figsize=(6, 6), layout="constrained")
        for ax_i in ax:
            ax_i.fill_between(np.arange(-50, xlim[-1] * 1.5), 0.9, 1.1, color="lightgray", alpha=0.35)
            ax_i.axhline(1.0, color="k", linestyle="--")
            ax_i.grid(color="lightgray", linestyle=":", linewidth=0.7)
            ax_i.set_axisbelow(True)
            ax_i.set_xlim(xlim)

        with open(filename, "r") as f:
            data = json.load(f)
        _to_numpy_arrays(data)
        _add_derived_quantities(data)
        data = sorted(data, key=_sort_key)

        _plot_panel(ax[0], data, filename, "S", nu, emm, model_specs, 0)
        _plot_panel(ax[1], data, filename, "E", nu, emm, model_specs, 0)

        ax[0].set_ylabel(rf"${omega_symbol}^{{0, S}}/\Omega_H$")
        ax[1].set_ylabel(rf"${omega_symbol}^{{0, M}}/\Omega_H$")
        ax[1].set_xlabel(r"$\left[\dot{r}^2 / (r \dot{\varphi} )^2 \right]_{\dot{X} = 0}$")

        for any_ax in ax:
            any_ax.set_yscale(
                "function",
                functions=(lambda x: np.arcsinh((x - 1.0) / 0.3), lambda x: 1.0 + 0.3 * np.sinh(x)),
            )
            any_ax.set_xscale("asinh", linear_width=1.0e-3)
            any_ax.set_xticks([0] + [10**i for i in range(-3, int(np.log10(xlim[-1])) + 1)])
            any_ax.xaxis.set_minor_locator(
                mticker.AsinhLocator(linear_width=1.0e-3, base=10, subs=list(range(2, 10)), numticks=50)
            )

        marker = "X" if orbit == "eccentric" else "<"
        legend_handles = [
            plt.scatter([], [], color="k", marker="", label=orbit.capitalize()),
            plt.Line2D([], [], color="w", marker="", label=r"$m = {}$".format(",".join(m_str))),
        ]
        legend_handles.extend(
            plt.scatter([], [], color=color, marker="o", label=label) for _, color, label in model_specs
        )
        legend_handles.extend(
            [
                plt.scatter([], [], color="w", marker=marker, label=r"$\dot{r} > 0$", edgecolor="k"),
                plt.scatter([], [], color="k", marker=marker, label=r"$\dot{r} < 0$"),
            ]
        )
        fig.legend(handles=legend_handles, loc="outside upper center", ncols=4, frameon=False)
        plt.show()


## Figure 6


In [ ]:
# (2,1) energy flux for a = 0.3, e = 0.5, p = 10 and the tidal frequencies that predict its zeros
sim = basedir + "eccentric/teuk_a0.3000_e0.5000_p010.0000/"
abh, emm = 0.3, [1]
teuk = plottk.PlotterTKO(a=abh, path_dyn=sim, path_tk=sim)
teuk.load_dyn()
dyn = teuk.dyn
omgH = ut.get_omgH(abh)

rdot = ut.get_vr(dyn["r"], dyn["prstar"], dyn["pph"], abh)
omgT_E = dyn["Omg"] + 3.0 * rdot**2 / (dyn["Omg"] * dyn["r"] ** 2)
omgT_E_ns = ut.OmegaTE(dyn["r"], dyn["Omg"], rdot, 1.0e-3, abh, model="ns", emm=emm)
t_peak = dyn["t"][np.argmin(dyn["r"])]


def crossing_times(y, t, level):
    """Times at which y crosses level (midpoint of the two samples around the crossing)."""
    j_zero = []
    for n, (_, vals) in enumerate(groupby(range(len(y)), key=lambda j: y[j] > level)):
        if n > 0:
            j_zero.append(list(vals)[0])
    return [0.5 * (t[j - 1] + t[j]) for j in j_zero]


t_zeros = {"omg": crossing_times(dyn["Omg"], dyn["t"], omgH)}
for omgT, label in zip([omgT_E, omgT_E_ns], ["E", "Ens"]):
    if max(omgT) >= omgH > 0.0:
        t_zeros[label] = crossing_times(omgT, dyn["t"], omgH)

flux = teuk.integrate_hflx(nx=3601, ny=161, cfl=2.0, m=emm)
t_flux = flux["t"] + ut.adv_time_in(abh, dyn["r"][0])
j_good = (t_flux > dyn["t"][0]) & (t_flux < dyn["t"][-1])
t_flux = t_flux[j_good]
h_flux = flux["sum"]["E"]["dy"][j_good]
t_fine = np.linspace(t_flux[0], t_flux[-1], 10 * len(t_flux))
t_E_zero = crossing_times(np.interp(t_fine, t_flux, h_flux), t_fine, 0.0)

fig, ax = plt.subplots(2, 1, layout="constrained", figsize=(5.75, 6), sharex=True)
ax[1].plot(t_flux, h_flux, lw=1.5, color="k", label=r"$\dot{M}$", zorder=100)
ax[1].axhline(0.0, lw=1.5, color="k", ls=":")

ax[0].plot(dyn["t"], dyn["Omg"], lw=1.5, color="#332288", label=r"$\Omega$")
ax[0].plot(dyn["t"], omgT_E, lw=1.5, color="#44AA99", label=r"$\Omega_{\rm T, 21}^{M, dt}$")
ax[0].plot(dyn["t"], omgT_E_ns, lw=1.5, color="#88CCEE", label=r"$\Omega_{\rm T, 21}^{M, \rm NS, dt}$")
ax[0].axhline(omgH, lw=1.5, color="k", ls="--", label=r"$\Omega_{\rm H}$")

for any_ax in ax:
    for key, col in zip(["omg", "E", "Ens"], ["#332288", "#44AA99", "#88CCEE"]):
        for val in t_zeros.get(key, []):
            any_ax.axvline(val, lw=1.5, ls="-.", color=col)
    for tez in t_E_zero:
        any_ax.axvline(tez, lw=1.5, ls="--", color="k")
    any_ax.axvline(t_peak, lw=1.5, ls=":", color="k")
    any_ax.grid(color="lightgray", linestyle=":", linewidth=0.7)

ax[1].set_yscale("asinh", linear_width=1.0e-11)
ax[1].set_xlim([409, 819])
ax[1].set_xlabel(r"$t/M$")
ax[0].set_ylabel(r"$\Omega$")
ax[1].set_ylabel(r"$\dot{M}_{21}$")
ax[0].legend(frameon=True, loc="best", fontsize=12, facecolor="white", edgecolor="white", framealpha=0.85)
ax[1].set_yticks([-1.0e-8, -1.0e-9, -1.0e-10, 0.0, 1.0e-10, 1.0e-9])
plt.show()


## Figure 7

The quasi-circular instantaneous fluxes are compared against frequency-domain results that are not part of this release, so this figure is not reproduced here.


## Figures 8 and 9

Set `use_eccentric = False` to produce Figure 9 (hyperbolic).


In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.ticker import MaxNLocator, LogLocator, NullFormatter
from matplotlib.lines import Line2D

import json


def load_fig78_model_data(filepath):
    with np.load(filepath, allow_pickle=False) as payload:
        manifest = json.loads(payload["manifest"].item())
        cases = {
            entry["folder"]: {
                "kind": entry["kind"],
                "edot_models": payload[f"{entry['key']}__edot_models"],
                "sdot_models": payload[f"{entry['key']}__sdot_models"],
            }
            for entry in manifest["entries"]
        }
    return {"model_names": manifest["model_names"], "cases": cases}


class SparseAsinhLocator(mticker.Locator):
    """
    Major ticks for an asinh axis: 0 and powers of 10 only, thinned so that
    neighbouring ticks are >= mingap (fraction of axis height) apart,
    with at most maxticks ticks.
    """

    def __init__(self, linear_width, maxticks=4, mingap=0.2):
        self.linear_width = linear_width
        self.maxticks = maxticks
        self.mingap = mingap

    def __call__(self):
        vmin, vmax = self.axis.get_view_interval()
        return self.tick_values(vmin, vmax)

    def _tr(self, x):
        return np.arcsinh(np.asarray(x) / self.linear_width)

    def tick_values(self, vmin, vmax):
        vmin, vmax = sorted((vmin, vmax))
        span = self._tr(vmax) - self._tr(vmin)
        kmin = int(np.floor(np.log10(self.linear_width)))
        kmax = int(np.floor(np.log10(max(abs(vmin), abs(vmax)))))
        cands = {sign * 10.0**k for k in range(kmin, kmax + 1) for sign in (1, -1)}
        # priority: 0 first, then largest magnitudes, so the outer ticks survive
        cands = [0.0] + sorted(cands, key=lambda x: -abs(x))
        ticks = []
        for c in cands:
            if not (vmin <= c <= vmax):
                continue
            if all(abs(self._tr(c) - self._tr(t)) >= self.mingap * span for t in ticks):
                ticks.append(c)
            if len(ticks) == self.maxticks:
                break
        return np.sort(ticks)


def value_from_folder(folder, token):
    return float(folder.split(token)[1].split("_")[0])


def build_fig78_grid(folds, use_eccentric):
    row_token = "_e" if use_eccentric else "_E"
    combos = {
        (value_from_folder(folder, row_token), value_from_folder(folder, "teuk_a")): (folder, lims)
        for folder, lims in folds
    }
    row_values = sorted({row for row, _ in combos})
    a_values = sorted({a for _, a in combos})
    return combos, row_values, a_values


def nearest_index(array, target_value):
    return int(np.abs(np.asarray(array) - target_value).argmin())


def padded_asinh_limits(reference_series, model_series, idx_start, idx_stop, pad=1.3):
    curves = [reference_series[idx_start:idx_stop]] + [series[idx_start:idx_stop] for series in model_series]
    ymax = max(np.max(arr) for arr in curves)
    ymin = min(np.min(arr) for arr in curves)
    return (
        ymin * pad if ymin < 0 else ymin / pad,
        ymax * pad if ymax > 0 else ymax / pad,
    )


def format_nonzero_tick(x, _):
    if x == 0 or abs(x) <= 1e-11:
        return ""
    exp = int(np.floor(np.log10(abs(x))))
    coeff = x / 10**exp
    if abs(coeff - round(coeff)) < 1e-10:
        coeff = int(round(coeff))
    if coeff == 1:
        return r"$10^{" + str(exp) + r"}$"
    if coeff == -1:
        return r"$-10^{" + str(exp) + r"}$"
    return r"$" + str(coeff) + r"\\times 10^{" + str(exp) + r"}$"


def plot_model_family(ax, t_num, y_num, t_model, model_series):
    ax.plot(t_num, y_num, lw=2.0, color="black")
    for idx, series in enumerate(model_series):
        ax.plot(t_model, series, color=colors[idx], lw=1.0, ls="--")


def plot_relative_errors(ax, t_model, model_series, ref_series):
    for idx, series in enumerate(model_series):
        ax.plot(t_model, abs((series - ref_series) / ref_series), color=colors[idx], lw=1.3)

In [ ]:
# load the precomputed analytical model curves for Figures 7 and 8
fig78_model_data = load_fig78_model_data("../dat/fig78_model_data.npz")
print(len(fig78_model_data["cases"]), "cases loaded")
print(fig78_model_data["model_names"])

In [ ]:
use_eccentric = True

# to use the hyperbolic set, set use_eccentric to False
_a_values = [-0.3, 0.0, 0.3, 0.8]

_e_values = [0.2, 0.4, 0.6]
_ecc_lims_by_a = {
    -0.3: [[450, 1000], [450, 1000], [550, 1200]],
    0.0: [[450, 1000], [450, 1000], [550, 1200]],
    0.3: [[450, 1000], [450, 1000], [550, 800]],
    0.8: [[450, 1000], [450, 1000], [400, 900]],
}
eccentric_set = [
    (f"/eccentric/teuk_a{a:.4f}_e{e:.4f}_p09.0000", _ecc_lims_by_a[a][idx])
    for a in _a_values
    for idx, e in enumerate(_e_values)
]

# hyperbolic runs: larger r0 than the original 100 M where available, to separate the
# encounter from the initial junk radiation (time windows in M)
hyperbolic_set = [
    ("/hyperbolic/teuk_a-0.3000_E1.0350_L8.0000_r0100.0000", [150, 550]),
    ("/hyperbolic/teuk_a-0.3000_E1.0650_L8.0000_r0300.0000", [680, 1030]),
    ("/hyperbolic/teuk_a-0.3000_E1.0950_L8.0000_r0400.0000", [820, 1120]),
    ("/hyperbolic/teuk_a0.0000_E1.0350_L8.0000_r0100.0000", [150, 550]),
    ("/hyperbolic/teuk_a0.0000_E1.0650_L8.0000_r0300.0000", [665, 1015]),
    ("/hyperbolic/teuk_a0.0000_E1.0950_L8.0000_r0100.0000", [100, 400]),
    ("/hyperbolic/teuk_a0.3000_E1.0350_L8.0000_r0100.0000", [150, 550]),
    ("/hyperbolic/teuk_a0.3000_E1.0650_L8.0000_r0300.0000", [670, 1020]),
    ("/hyperbolic/teuk_a0.3000_E1.0950_L8.0000_r01200.0000", [2770, 3070]),
    ("/hyperbolic/teuk_a0.8000_E1.0350_L8.0000_r0100.0000", [150, 550]),
    ("/hyperbolic/teuk_a0.8000_E1.0650_L8.0000_r0300.0000", [670, 1020]),
    ("/hyperbolic/teuk_a0.8000_E1.0950_L8.0000_r0400.0000", [820, 1120]),
]


In [ ]:
# choose whether to do eccentric or hyperbolic
folds = eccentric_set if use_eccentric else hyperbolic_set
combos, row_values, a_values = build_fig78_grid(folds, use_eccentric)

nrows = len(row_values)
ncols = len(a_values)
model_names = fig78_model_data["model_names"]

# Single figure: energy and angular momentum fluxes
top_margin = 0.95
bottom_margin = 0.05
left_margin = 0.12
_gs_kw = dict(
    hspace=0.08,
    wspace=0.24,
    top=top_margin,
    bottom=bottom_margin,
    left=left_margin,
    right=0.985,
)
fig = plt.figure(figsize=(2.8 * ncols, 4.8 * nrows))
outer_gs = GridSpec(nrows, ncols, figure=fig, **_gs_kw)

for i, row_val in enumerate(row_values):
    for j, a_val in enumerate(a_values):
        case = combos.get((row_val, a_val))
        if case is None:
            continue

        folder, (xmin, xmax) = case

        igs = GridSpecFromSubplotSpec(
            4,
            1,
            subplot_spec=outer_gs[i, j],
            height_ratios=[1, 1, 1, 1],
            hspace=0.13,
        )

        axm = fig.add_subplot(igs[0])
        axrm = fig.add_subplot(igs[1], sharex=axm)
        axs = fig.add_subplot(igs[2], sharex=axm)
        axrs = fig.add_subplot(igs[3], sharex=axm)

        flux_axes = (axm, axs)
        error_axes = (axrm, axrs)
        all_axes = flux_axes + error_axes

        plt.setp(axm.get_xticklabels(), visible=False)
        plt.setp(axrm.get_xticklabels(), visible=False)
        plt.setp(axs.get_xticklabels(), visible=False)

        if i == 0:
            axm.set_title(r"$\hat{{a}} = {:.1f}$".format(a_val), fontsize=12, pad=3)

        abh = value_from_folder(folder, "teuk_a")
        pltk = plottk.PlotterTKO(abh, basedir + folder, basedir + folder)
        pltk.load_dyn()
        res = max(pltk.sims, key=lambda k: len(pltk.sims[k]))
        hflx = pltk.integrate_hflx(nx=res[0], ny=res[1], cfl=res[2])
        v0 = ut.adv_time_in(abh, pltk.dyn["r"][0])

        edot_num = hflx["sum"]["E"]["dy"]
        sdot_num = hflx["sum"]["J"]["dy"]

        print(f"Loaded data for a={a_val}, row_val={row_val}")

        case_models = fig78_model_data["cases"].get(folder)
        if case_models is None:
            raise KeyError(f"No precomputed Figure 7/8 model data found for {folder}")

        edot_models = case_models["edot_models"]
        sdot_models = case_models["sdot_models"]

        t_dyn = pltk.dyn["t"]
        n_time = len(t_dyn)
        for name, model_array in (("edot", edot_models), ("sdot", sdot_models)):
            if model_array.shape[1] != n_time:
                raise ValueError(
                    f"Precomputed {name} model length mismatch for {folder}: "
                    f"{model_array.shape[1]} != {n_time}"
                )

        t_num = pltk.hflx["t"] + v0
        edot_on_dyn = np.interp(t_dyn, t_num, edot_num)
        sdot_on_dyn = np.interp(t_dyn, t_num, sdot_num)

        plot_model_family(axm, t_num, edot_num, t_dyn, edot_models)
        plot_model_family(axs, t_num, sdot_num, t_dyn, sdot_models)
        plot_relative_errors(axrm, t_dyn, edot_models, edot_on_dyn)
        plot_relative_errors(axrs, t_dyn, sdot_models, sdot_on_dyn)

        for tax in error_axes:
            tax.axhline(1e-1, ls="--", color="gray", lw=1.0)
            tax.set_yscale("log")
            tax.yaxis.set_major_locator(LogLocator(numticks=4))
            tax.yaxis.set_minor_formatter(NullFormatter())

        for tax in flux_axes:
            tax.set_yscale("asinh", linear_width=1e-10)
            tax.yaxis.set_major_locator(SparseAsinhLocator(1e-10, maxticks=12, mingap=0.1))
            tax.yaxis.set_major_formatter(format_nonzero_tick)

        for tax in all_axes:
            tax.xaxis.set_major_locator(MaxNLocator(nbins=4, prune="both"))
            tax.tick_params(labelsize=8)

        if j == 0:
            for ax, label, size in (
                (axm, r"$\dot{M}$", 10),
                (axrm, r"$|\Delta \dot{M}/\dot{M}|$", 9),
                (axs, r"$\dot{S}$", 10),
                (axrs, r"$|\Delta \dot{S}/\dot{S}|$", 9),
            ):
                ax.set_ylabel(label, fontsize=size)

        if i == nrows - 1:
            axrs.set_xlabel(r"$t/M$", fontsize=10)

        idx_start = nearest_index(t_dyn, xmin)
        idx_stop = nearest_index(t_dyn, xmax)
        axm.set_ylim(*padded_asinh_limits(edot_on_dyn, edot_models, idx_start, idx_stop))
        axs.set_ylim(*padded_asinh_limits(sdot_on_dyn, sdot_models, idx_start, idx_stop))

        for tax in all_axes:
            tax.set_xlim(xmin, xmax)
            tax.grid(alpha=0.4)

        for tax in error_axes:
            if tax.get_ylim()[1] > 4e2:
                tax.set_ylim(top=4e2)
            if tax.get_ylim()[0] < 1e-4:
                tax.set_ylim(bottom=1e-4)

# Shared legend and row labels for both figures
model_handles = [Line2D([0], [0], color="black", lw=2.0, label="Numerical")] + [
    Line2D([0], [0], color=colors[k], lw=1.8, label=f"Model {k+1}")
    for k in range(len(model_names))
]
_leg_kw = dict(
    loc="upper center",
    ncol=6,
    fontsize=14,
    frameon=False,
    bbox_to_anchor=(0.5, 1.0),
)
dy = (top_margin - bottom_margin) / nrows
_leg = fig.legend(handles=model_handles, **_leg_kw)
fig.add_artist(_leg)
for i, row_val in enumerate(row_values):
    yrow = top_margin - (i + 0.5) * dy
    row_label = (
        r"$e_0 = {:.3f}$".format(row_val)
        if use_eccentric
        else r"$E_0 = {:.3f}$".format(row_val)
    )
    fig.text(
        left_margin - 0.09,
        yrow,
        row_label,
        rotation=90,
        va="center",
        ha="center",
        fontsize=11,
    )

plt.show()

## Figure 10


In [ ]:
data_fname = "../dat/data.txt"

with open(data_fname, "r") as f:
    all_data = np.genfromtxt(f, names=True)

data_subsets = [all_data[all_data["kind"] == orbit_type] for orbit_type in (0, 1, 2)]
orders = ["fujita"]

# 4 columns: a0, rmin, omega_max, histograms
fig, ax = plt.subplots(
    2,
    4,
    sharex="col",
    sharey="row",
    figsize=(15, 6),
    gridspec_kw={"width_ratios": [1, 1, 1, 2 / 3]},
)

bins = np.logspace(-4, 2, 30)
scatter_x_keys = ("a", "rmin", "omg_max")
orbit_kinds = ("circular", "eccentric", "hyperbolic")

for order in orders:
    for subset, kind in zip(data_subsets, orbit_kinds):
        delta_E = np.abs((subset["dEavg"] - subset[f"dEpn_{order}"]) / subset["dEavg"])
        delta_J = np.abs((subset["dJavg"] - subset[f"dJpn_{order}"]) / subset["dJavg"])

        this_label = kind.capitalize()
        color, *_ = colors_orbit_types[kind]

        for row, delta in enumerate((delta_E, delta_J)):
            for col, x_key in enumerate(scatter_x_keys):
                ax[row, col].scatter(
                    subset[x_key],
                    delta,
                    label=this_label if (row == 0 and col == 0) else None,
                    alpha=0.7,
                    color=color,
                    edgecolors="white",
                    linewidths=0.4,
                )
                ax[row, col].axhline(
                    np.median(delta),
                    color=color,
                    linestyle="--",
                    linewidth=1.2,
                    alpha=0.8,
                )

            ax[row, 3].hist(
                delta,
                bins=bins,
                orientation="horizontal",
                alpha=0.5,
                color=color,
                label=this_label if row == 0 else None,
            )

# --- Scatter columns: grid + shaded band ---
for col in (0, 1, 2):
    for tax in ax[:, col]:
        tax.set_yscale("log")
        tax.grid(color="lightgray", linestyle=":", linewidth=0.7)
        tax.fill_between([-1e6, 1e6], 2e-4, 1e-2, color="lightgray", alpha=0.25)

for tax in ax[:, 2]:
    tax.set_xscale("log")

for tax in ax[:, 1]:
    tax.set_xscale("log")
    tax.set_xticks([3, 10, 100])
    tax.set_xticklabels(["3", "10", ""])
    tax.tick_params(axis="x", rotation=0)

# --- Histogram column: frameless style ---
for tax in ax[:, 3]:
    tax.set_yscale("log")
    tax.grid(color="lightgray", linestyle=":", linewidth=0.7, axis="y")

    for spine in tax.spines.values():
        spine.set_visible(False)

    tax.tick_params(axis="x", bottom=False, labelbottom=False)
    tax.tick_params(axis="y", left=False, labelleft=False)

# --- Axis labels ---
ax[1, 0].set_xlabel(r"$\hat{a}$", fontsize=15)
ax[1, 1].set_xlabel(r"$r_{\rm min}$", fontsize=15)
ax[1, 2].set_xlabel(r"$\Omega_{\rm max}$", fontsize=15)

ax[0, 0].set_ylabel(r"$|\Delta \dot{M} / \dot{M}|$", fontsize=15)
ax[1, 0].set_ylabel(r"$|\Delta \dot{S} / \dot{S}|$", fontsize=15)

# --- Y/x limits ---
for tax in ax.flatten():
    tax.set_ylim(2e-4, 1e1)

ax[1, 0].set_xlim(-0.99, 1.1)
ax[1, 1].set_xlim(3.0, 100.0)
ax[1, 2].set_xlim(2e-4, 1.0)

plt.subplots_adjust(
    wspace=0.05, hspace=0.1, left=0.08, right=0.97, top=0.9, bottom=0.1
)

# Attach histogram panels flush against the omega_max scatter column
fig.canvas.draw()
for row in range(2):
    pos_scatter = ax[row, 2].get_position()
    pos_hist = ax[row, 3].get_position()
    ax[row, 3].set_position(
        [pos_scatter.x1, pos_hist.y0, pos_hist.width, pos_hist.height]
    )

# Centre legend above the three scatter columns
fig.canvas.draw()
handles, labels = ax[0, 3].get_legend_handles_labels()
# midpoint in figure coords between left edge of col0 and right edge of col2
x0 = ax[0, 0].get_position().x0
x1 = ax[0, 2].get_position().x1
mid = (x0 + x1) / 2
top = ax[0, 0].get_position().y1
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(mid, top + 0.01),
    bbox_transform=fig.transFigure,
    ncol=3,
    frameon=False,
    fontsize=14,
)
plt.show()

In [ ]:
from scipy.stats import spearmanr
for subset, kind in zip(data_subsets, ["circular", "eccentric", "hyperbolic"]):
    dE = np.abs((subset["dEavg"] - subset[f"dEpn_{order}"]) / subset["dEavg"])
    dJ = np.abs((subset["dJavg"] - subset[f"dJpn_{order}"]) / subset["dJavg"])
    mask_strong = subset["rmin"] < 10

    print(f"\n=== {kind} (N={len(subset)}) ===")
    print(f"  median dE={np.median(dE)*100:.1f}%  dJ={np.median(dJ)*100:.1f}%")
    if mask_strong.sum() > 0:
        print(
            f"  rmin<10 (N={mask_strong.sum()}): "
            f"dE={np.median(dE[mask_strong])*100:.1f}%  "
            f"dJ={np.median(dJ[mask_strong])*100:.1f}%"
        )
    print(f"  <10%:  dE={( dE<0.1).mean()*100:.0f}%  dJ={(dJ<0.1).mean()*100:.0f}%")
    print(f"  <50%:  dE={(dE<0.5).mean()*100:.0f}%  dJ={(dJ<0.5).mean()*100:.0f}%")

# Spearman correlations on full dataset
all_dE = np.abs((all_data["dEavg"] - all_data[f"dEpn_{order}"]) / all_data["dEavg"])
all_dJ = np.abs((all_data["dJavg"] - all_data[f"dJpn_{order}"]) / all_data["dJavg"])

print("\n=== Spearman correlations (all orbits) ===")
for col in ["omg_max", "rmin", "a"]:
    rho_E, _ = spearmanr(all_data[col], all_dE)
    rho_J, _ = spearmanr(all_data[col], all_dJ)
    print(f"  {col:10s}  rho(dE)={rho_E:+.3f}  rho(dJ)={rho_J:+.3f}")

# Per-orbit-type Spearman for omg_max
print("\n=== Spearman omg_max per orbit type ===")
for subset, kind in zip(data_subsets, ["circular", "eccentric", "hyperbolic"]):
    dE = np.abs((subset["dEavg"] - subset[f"dEpn_{order}"]) / subset["dEavg"])
    dJ = np.abs((subset["dJavg"] - subset[f"dJpn_{order}"]) / subset["dJavg"])
    if len(subset) < 4:
        continue
    rho_E, _ = spearmanr(subset["omg_max"], dE)
    rho_J, _ = spearmanr(subset["omg_max"], dJ)
    print(f"  {kind:12s}  rho(dE)={rho_E:+.3f}  rho(dJ)={rho_J:+.3f}")

## Figure 11

Contribution of the individual $m$ modes, relative to $m=2$.


In [ ]:
configs = [
    "circular/teuk_a0.9000_r04.0000_circ",
    "eccentric/teuk_a0.2700_e0.6000_p06.3000",
    "hyperbolic/teuk_a0.9000_E1.0050_L4.0000_r0150.0000",
]
mode_styles = [(0, ":"), (1, "-."), (2, "-"), (3, "--"), (4, (0, (3, 1, 1, 1)))]

fig, ax = plt.subplots(2, 2, sharex=True, figsize=(12, 6))
for cfg, col in zip(configs, colors):
    sim = basedir + cfg
    abh = float(cfg.split("teuk_a")[1].split("_")[0])
    ptk = plottk.PlotterTKO(a=abh, path_dyn=sim, path_tk=sim)
    ptk.load_dyn()
    res = max(ptk.sims, key=lambda k: len(ptk.sims[k]))
    hflx = ptk.integrate_hflx(nx=res[0], ny=res[1], cfl=res[2])

    t_num = hflx["t"] + ut.adv_time_in(ptk.a, ptk.dyn["r"][0])
    t_norm = t_num / np.max(t_num)
    for mv, ls in mode_styles:
        if mv not in hflx:
            continue
        if mv == 2:
            ax[0, 0].plot(t_norm, hflx[2]["E"]["dy"], color=col, ls=ls)
            ax[0, 1].plot(t_norm, hflx[2]["J"]["dy"], color=col, ls=ls)
            continue
        ax[1, 0].plot(t_norm, abs(hflx[mv]["E"]["dy"] / hflx[2]["E"]["dy"]), color=col, ls=ls)
        if mv != 0:
            ax[1, 1].plot(t_norm, abs(hflx[mv]["J"]["dy"] / hflx[2]["J"]["dy"]), color=col, ls=ls)

tks = [10.0**k for k in range(-11, 0)]
for tax in ax[1, :]:
    tax.set_yscale("log")
    tax.set_yticks(ticks=tks, minor=True, labels=["" for _ in tks])
    tax.grid(color="lightgray", ls=":")
    tax.set_xlabel(r"$t/t_{\rm max}$")
    tax.set_xlim(0.1, 0.9)
ax[0, 0].set_yscale("asinh", linear_width=1.0e-9)
ax[0, 1].set_yscale("asinh", linear_width=1.0e-7)
for tax in ax[0, :]:
    tax.grid(color="lightgray", ls=":")
ax[0, 0].set_ylabel(r"$\dot{M}_{2}$", fontsize=15)
ax[0, 1].set_ylabel(r"$\dot{S}_{2}$", fontsize=15)
ax[1, 0].set_ylabel(r"$\dot{M}_{m}/\dot{M}_{2}$", fontsize=15)
ax[1, 1].set_ylabel(r"$\dot{S}_{m}/\dot{S}_{2}$", fontsize=15)
ax[1, 0].set_ylim(1e-10, 1e4)

from matplotlib.lines import Line2D

first_legend = ax[0, 0].legend(
    handles=[Line2D([0], [0], color=c, lw=2, label=l) for c, l in zip(colors, ["Circular", "Eccentric", "Hyperbolic"])],
    loc="upper center", bbox_to_anchor=(1, 1.35), ncol=3, frameon=False, fancybox=False,
)
ax[0, 0].add_artist(first_legend)
ax[0, 0].legend(
    handles=[Line2D([0], [0], color="k", lw=2, ls=ls, label=rf"$m={mv}$") for mv, ls in mode_styles],
    loc="upper center", bbox_to_anchor=(1, 1.25), ncol=5, frameon=False, fancybox=False,
)
plt.show()


## Figure 12

Convergence of the $m=2$ fluxes with the number of radial points $n_x$ (high, medium, low resolution $n_x = 7201, 3601, 1801$), with the medium–low difference rescaled by the factor $\mathcal{C}(p)$ expected for convergence order $p = 2$.


In [ ]:
configs = [
    "circular/teuk_a0.9000_r04.0000_circ",
    "eccentric/teuk_a0.2700_e0.6000_p06.3000",
    "hyperbolic/teuk_a0.9500_E1.3100_L3.2900_r0200.0000",
]
p = 2
nx = [1801, 3601, 7201]

fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for cfg, col in zip(configs, colors):
    sim = basedir + cfg
    abh = float(cfg.split("teuk_a")[1].split("_")[0])
    ptk = plottk.PlotterTKO(a=abh, path_dyn=sim, path_tk=sim)
    ptk.load_dyn()
    xH = ut.compute_xH(abh, Shh=10.0)
    dv = ut.adv_time_in(ptk.a, ptk.dyn["r"][0])

    conv = {}
    for x in nx:
        hflx = ptk.integrate_hflx(nx=x, ny=161, m=2)
        conv[x] = {
            "hx": (10.0 - xH) / x,
            "v": hflx["t"] + dv,
            "dE": hflx[2]["E"]["dy"],
            "dJ": hflx[2]["J"]["dy"],
        }

    x0, x1, x2 = nx
    v_int = np.linspace(
        max(conv[x]["v"][0] for x in nx), min(conv[x]["v"][-1] for x in nx), 1000
    )
    scale = abs(
        (conv[x2]["hx"] ** p - conv[x1]["hx"] ** p) / (conv[x1]["hx"] ** p - conv[x0]["hx"] ** p)
    )
    for quantity, axis in zip(["dE", "dJ"], ax):
        q0, q1, q2 = (np.interp(v_int, conv[x]["v"], conv[x][quantity]) for x in nx)
        axis.plot(v_int / max(v_int), np.abs(q2 - q1), linewidth=1.5, color=col)
        axis.plot(v_int / max(v_int), np.abs(q1 - q0) * scale, linestyle="--", linewidth=1.5, color=lighten_color(col, 1.3))

for axis, label in zip(ax, [r"$\dot{M}_2$", r"$\dot{S}_2$"]):
    axis.set_yscale("log")
    axis.set_ylabel(label)
    axis.grid(color="lightgray", linestyle=":", linewidth=0.7)
    axis.set_ylim(1e-20, 1e-3)

first_legend = ax[0].legend(
    handles=[plt.Line2D([], [], color=c, label=f"{k}, $p$ = {p}") for c, k in zip(colors, ["Circular", "Eccentric", "Hyperbolic"])],
    loc="upper center", bbox_to_anchor=(0.5, 1.3), ncol=3, frameon=False, fancybox=False,
)
ax[0].add_artist(first_legend)
ax[0].legend(
    handles=[
        plt.Line2D([], [], color="k", linestyle="-", label=r"$|Q^{\rm H} - Q^{\rm M}|$"),
        plt.Line2D([], [], color="k", linestyle="--", label=r"$\mathcal{C}(p)|Q^{\rm M} - Q^{\rm L}|$"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=3, frameon=False, fancybox=False,
)
ax[1].set_xlabel(r"$t/t_{\rm max}$")
ax[1].set_xlim(0.0, 1)
plt.show()
